# Databricks SQL Workshop

**Dataset:** 400 Drivers | 400 Taxis | 5 Taxi Types



In [0]:
df = spark.table("workspace.rebu.passengers")
display(df)

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

---

#  Setup & Temporary Views

## Dataset Overview

This tutorial uses three tables from the REBU Singapore taxi system:

- **drivers** — 400 rows, 11 columns (DriverID, Age, Gender, YearsExperience, Rating, JoinDate, etc.)
- **taxi** — 400 rows, 6 columns (TaxiID, LicensePlate, TaxiTypeID, Model, YearManufactured, Status)
- **taxi_type** — 5 rows, 5 columns (TaxiTypeID, TypeName, BaseFare, PerKmRate, Capacity)

---

### Key Concept: Unity Catalog & Temporary Views

In Databricks, every SQL query runs against tables or views in the **Unity Catalog**. 

**Temporary views** let you give a DataFrame or filtered subset a name that can be queried with SQL during your session. They:
- Exist only for your current session
- Store the query definition, NOT the data
- Re-execute each time you query them
- Disappear when the session ends

# SQL Excercise
## Explanation: 

USE Statement
```sql
USE database_name.schema_name;
```

**Components:**
- `USE`: Sets the default database/schema context
- **Purpose**: Allows you to reference tables by short name instead of fully qualified names
- **Scope**: Session-level setting (lasts until you change it or session ends)
- **Example**: After `USE workspace.rebu`, you can write `SELECT * FROM drivers` instead of `SELECT * FROM workspace.rebu.drivers`

In [0]:
%sql


-- ============================================================
-- Set the default schema so we can reference tables by short name
-- ============================================================
USE workspace.rebu;

-- Quick check: how many rows in each table?
SELECT 'drivers' AS tbl, COUNT(*) AS rows FROM drivers
UNION ALL
SELECT 'taxi', COUNT(*) FROM taxi
UNION ALL
SELECT 'taxi_type', COUNT(*) FROM taxi_type
ORDER BY rows DESC;


### Syntax:

**`UNION ALL`**: Combines results from multiple SELECT statements
- Unlike `UNION`, it keeps duplicate rows (faster)
- All SELECT statements must have same number of columns
- Column types must be compatible

**`COUNT(*)`**: Counts all rows including NULLs
- Use `COUNT(column_name)` to exclude NULLs
- Use `COUNT(DISTINCT column)` to count unique values

**`ORDER BY`**: Sorts the final result set
- `DESC` = descending (highest to lowest)
- `ASC` = ascending (lowest to highest) - default

## Explanation: 

CREATE TEMP VIEW

```sql
CREATE [OR REPLACE] TEMP VIEW view_name AS
SELECT ...
FROM table_name
WHERE conditions;
```

**Components:**
- `CREATE TEMP VIEW`: Creates a session-scoped virtual table
- `OR REPLACE`: If view exists, replace it (prevents errors)
- `TEMP`/`TEMPORARY`: View exists only in current session/notebook
- **Virtual**: Stores query definition, not data (re-executes each time)
- **Use Case**: Perfect for creating reusable filtered subsets

**Key Difference from Regular Views:**
- Regular views persist in the metastore
- Temp views disappear when session ends
- Temp views are faster to create (no metadata operations)

In [0]:
%sql
-- Create a temporary view of elite drivers
-- CREATE TEMPORARY VIEW — a session-scoped named query
-- Temp views exist only for the current session/notebook.
-- They are perfect for creating reusable filtered subsets.
-- Create a temp view of experienced, highly-rated drivers

CREATE OR REPLACE TEMP VIEW elite_drivers AS
SELECT
    DriverID,
    FirstName,
    LastName,
    Age,
    Gender,
    YearsExperience,
    Rating,
    JoinDate
FROM drivers
WHERE Rating >= 4.7
  AND YearsExperience >= 10;

-- Now query the view like a regular table
SELECT COUNT(*) AS elite_count FROM elite_drivers;


### Syntax :

**`WHERE`**: Filters rows based on conditions
- Only rows meeting ALL conditions pass through
- Evaluated BEFORE SELECT clause
- Cannot use aggregate functions (use HAVING for that)

**`AND`**: Logical operator requiring BOTH conditions to be true
- Truth table: TRUE AND TRUE = TRUE, all others = FALSE
- Can chain multiple: `WHERE a AND b AND c`

**`>=`**: Greater than or equal to comparison operator
- Other operators: `=`, `!=` or `<>`, `<`, `>`, `<=`

## Explanation: 
JOIN

```sql
SELECT columns
FROM table1 alias1
JOIN table2 alias2 ON alias1.key = alias2.key;
```

**Components:**
- `JOIN` (or `INNER JOIN`): Combines rows from two tables based on related columns
- **Table Alias**: Short name (t, tt) for easier reference
- `ON condition`: Specifies how tables relate (matching keys)
- **Result**: Only rows where the condition is TRUE

**Types of JOINs:**
- `INNER JOIN`: Only matching rows (default)
- `LEFT JOIN`: All left table rows + matches from right (NULLs where no match)
- `RIGHT JOIN`: All right table rows + matches from left
- `FULL OUTER JOIN`: All rows from both tables
- `CROSS JOIN`: Cartesian product (every combination)

In [0]:
%sql
-- Create a temp view that enriches taxi with type labels
-- This pre-joined view simplifies all subsequent queries

CREATE OR REPLACE TEMP VIEW taxi_enriched AS
SELECT
    t.TaxiID,
    t.LicensePlate,
    t.Model,
    t.YearManufactured,
    t.Status,
    tt.TypeName AS taxi_type,
    tt.BaseFare,
    tt.PerKmRate
FROM taxi t
JOIN taxi_type tt ON t.TaxiTypeID = tt.TaxiTypeID;

SELECT * FROM taxi_enriched LIMIT 3;


### Syntax:

**`AS`**: Creates an alias (alternative name) for columns or tables
- Column alias: `SELECT col AS new_name`
- Table alias: `FROM table_name t`
- Makes queries more readable and concise

**`LIMIT n`**: Returns only the first n rows
- Useful for testing queries on large datasets
- Applied AFTER all other operations (ORDER BY, etc.)

**Dot notation** (`t.TaxiTypeID`): Accesses columns from specific table alias
- Required when same column name exists in multiple tables
- Best practice: always use table alias for clarity

#  SELECT Fundamentals

## Concept

**SELECT** is the foundation of every SQL query. Master column selection, aliases, expressions, and DISTINCT to control exactly what data you retrieve.

**Best Practice**: Always specify column names in production — never use `SELECT *`

**Why?**
- Performance: Only read needed columns from disk (column pruning)
- Clarity: Makes queries self-documenting
- Stability: Schema changes won't break your queries
- Efficiency: Reduces network transfer and memory usage

## Syntax: 

SELECT & Column Pruning
```sql
SELECT column1, column2, column3
FROM table_name
LIMIT n;
```

**Components:**
- `SELECT`: Specifies which columns to retrieve
- **Column Pruning**: Reading only needed columns (not all 11)
- **Performance Impact**: Delta/Parquet format stores data columnarly
  - Reading 5 columns reads ~5/11 of the data
  - `SELECT *` forces reading ALL columns
- **Best Practice**: List exactly the columns you need

**Query Execution Order** (what SQL actually does):
1. `FROM` - Identify table
2. `WHERE` - Filter rows
3. `GROUP BY` - Create groups
4. `HAVING` - Filter groups
5. `SELECT` - Choose columns
6. `ORDER BY` - Sort results
7. `LIMIT` - Restrict count

In [0]:
%sql

SELECT
    DriverID,
    FirstName,
    LastName,
    Age,
    Rating
FROM drivers
LIMIT 5;


## Explanation: 

Computed Columns & Functions

```sql
SELECT 
    column1,
    expression AS alias_name,
    FUNCTION(column) AS result_name
FROM table_name;
```

**Expressions**: Math operations on columns
- `+`, `-`, `*`, `/` : Basic arithmetic
- `%` : Modulo (remainder)
- Can combine multiple columns: `Age - YearsExperience`

**Common String Functions:**
- `CONCAT(str1, str2, ...)`: Combines text strings
- `UPPER(str)` / `LOWER(str)`: Change case
- `SUBSTRING(str, start, length)`: Extract portion
- `LENGTH(str)`: String length
- `TRIM(str)`: Remove leading/trailing spaces

**Common Math Functions:**
- `ROUND(value, decimals)`: Rounds to specified decimal places
- `CEIL(value)` / `FLOOR(value)`: Round up/down to integer
- `ABS(value)`: Absolute value
- `POWER(base, exponent)`: Exponentiation

**Date Functions:**
- `YEAR(date)`: Extract year
- `MONTH(date)`: Extract month
- `DAY(date)`: Extract day
- `CURRENT_DATE()`: Today's date
- `DATEDIFF(date1, date2)`: Days between dates

In [0]:
%sql

-- Computed columns with AS (aliases) and expressions
-- Create new columns from existing ones using math/functions
SELECT
    DriverID,
    FirstName,
    Age,
    YearsExperience,
    Age - YearsExperience AS age_when_started,
    ROUND(Rating * 20, 0) AS rating_pct,
    CONCAT(FirstName, ' ', LastName) AS full_name,
    YEAR(JoinDate) AS join_year
FROM drivers
LIMIT 5;


### Function Explanation:

**`ROUND(Rating * 20, 0)`**: 
- Multiplies rating by 20
- Rounds to 0 decimal places (integer)
- Converts 0-5 scale to 0-100 percentage

**`CONCAT(FirstName, ' ', LastName)`**:
- String concatenation
- Joins text with space separator
- Alternative: `FirstName || ' ' || LastName`

**`YEAR(JoinDate)`**:
- Date function extracting year component
- Returns integer (2019, 2020, etc.)
- Similar: `MONTH()`, `DAY()`, `HOUR()`, etc.

## Explanation: 

DISTINCT & CASE WHEN

### DISTINCT
```sql
SELECT DISTINCT column FROM table;
SELECT COUNT(DISTINCT column) FROM table;
```

**Purpose**: Returns only unique values, removing duplicates
- Works on single or multiple columns
- `DISTINCT col1, col2`: Unique combinations
- `COUNT(DISTINCT col)`: Counts unique values
- **Performance**: Can be expensive on large datasets (requires sorting/hashing)

### CASE WHEN (SQL's If-Then-Else)
```sql
CASE
    WHEN condition1 THEN result1
    WHEN condition2 THEN result2
    WHEN condition3 THEN result3
    ELSE default_result
END AS column_alias
```

**Components:**
- `CASE`: Starts the conditional expression
- `WHEN condition THEN result`: If-then branches
- `ELSE`: Default value if no conditions match (optional)
- `END`: Required to close CASE statement
- `AS`: Alias for the new column

**Evaluation Logic:**
1. Evaluates conditions from top to bottom
2. Returns the FIRST matching THEN result
3. If no conditions match, returns ELSE value (or NULL if no ELSE)
4. Stops evaluation after first match (short-circuit)

In [0]:
%sql

-- DISTINCT — remove duplicate values
-- How many distinct car models are in the fleet?
SELECT COUNT(DISTINCT Model) AS unique_models FROM taxi;


-- CASE WHEN — conditional logic inside SELECT
SELECT
    DriverID,
    FirstName,
    YearsExperience,
    CASE
        WHEN YearsExperience <= 5  THEN 'Junior'
        WHEN YearsExperience <= 15 THEN 'Mid-level'
        WHEN YearsExperience <= 25 THEN 'Senior'
        ELSE 'Expert'
    END AS experience_level,
    Rating
FROM drivers
LIMIT 6;


### CASE WHEN Logic Flow:

For the experience_level example:
1. If `YearsExperience <= 5`: Return 'Junior' (stop checking)
2. Else if `YearsExperience <= 15`: Return 'Mid-level' (stop checking)
3. Else if `YearsExperience <= 25`: Return 'Senior' (stop checking)
4. Else: Return 'Expert'

**Important**: Conditions are evaluated in order. A value of 3 years matches the first condition (≤5) and gets 'Junior', even though it also satisfies ≤15.

## Explanation: 

ORDER BY

```sql
SELECT columns
FROM table
ORDER BY column1 DESC, column2 ASC
LIMIT n;
```

**Components:**
- `ORDER BY`: Sorts result set by specified columns
- `DESC`: Descending order (highest to lowest, Z to A)
- `ASC`: Ascending order (lowest to highest, A to Z) - **default**
- **Multiple columns**: Second column breaks ties in first
- `NULLS FIRST` / `NULLS LAST`: Controls NULL value placement

**Sorting Rules:**
- Numbers: 1, 2, 3... (ASC) or 100, 99, 98... (DESC)
- Strings: Alphabetical (A, B, C... or Z, Y, X...)
- Dates: Chronological (oldest to newest or vice versa)
- NULLs: By default, NULLS LAST in ASC, NULLS FIRST in DESC

**Performance Note**: ORDER BY can be expensive on large datasets (requires full scan and sort)

In [0]:
%sql

-- ORDER BY — sort results ascending or descending
-- NULLS FIRST/LAST controls NULL placement

SELECT
    DriverID,
    FirstName,
    Rating,
    YearsExperience
FROM drivers
ORDER BY Rating DESC, YearsExperience DESC
LIMIT 5;


### Multi-Column Sorting:

In the query above:
1. First, sort by `Rating DESC` (highest ratings first)
2. When multiple drivers have the same Rating (e.g., 5.00), use `YearsExperience DESC` as tiebreaker
3. Result: Ng (5.00, 23 years) comes before Daniel (5.00, 9 years)

You can sort by as many columns as needed: `ORDER BY col1 DESC, col2 ASC, col3 DESC`

---

# WHERE Filtering

## Concept

**WHERE** filters rows BEFORE aggregation. It controls which rows enter the query pipeline.

**Key Points:**
- Evaluated BEFORE SELECT and GROUP BY
- Reduces data volume early (better performance)
- Cannot use aggregate functions (use HAVING for that)
- Can combine multiple conditions with AND, OR, NOT

**Execution Order:**
```
FROM → WHERE → GROUP BY → HAVING → SELECT → ORDER BY → LIMIT
```

## Explanation: 

WHERE Clause

```sql
SELECT columns
FROM table
WHERE condition
ORDER BY column;
```

**Comparison Operators:**
- `=` : Equal to
- `!=` or `<>` : Not equal to
- `>` : Greater than
- `>=` : Greater than or equal to
- `<` : Less than
- `<=` : Less than or equal to

**NULL Handling:**
- `IS NULL` : Check if value is NULL
- `IS NOT NULL` : Check if value is not NULL
- Cannot use `= NULL` or `!= NULL`

**Performance Impact:**
- WHERE reduces data early in pipeline
- Fewer rows = faster aggregation, sorting, etc.
- Always filter as early as possible

In [0]:
%sql
-- Basic WHERE with comparison operators
-- =, !=, <, >, <=, >=

SELECT DriverID, FirstName, Age, Rating
FROM drivers
WHERE Rating >= 4.9
ORDER BY Rating DESC
LIMIT 5;


## Explanation: 

Logical Operators

```sql
WHERE condition1 AND condition2
WHERE condition1 OR condition2  
WHERE NOT condition
WHERE (cond1 OR cond2) AND cond3
```

**Logical Operators:**

**`AND`**: BOTH conditions must be true
```
TRUE AND TRUE   = TRUE
TRUE AND FALSE  = FALSE
FALSE AND FALSE = FALSE
```

**`OR`**: At least ONE condition must be true
```
TRUE OR TRUE   = TRUE
TRUE OR FALSE  = TRUE
FALSE OR FALSE = FALSE
```

**`NOT`**: Negates/inverts a condition
```
NOT TRUE  = FALSE
NOT FALSE = TRUE
```

**Operator Precedence** (without parentheses):
1. NOT (highest)
2. AND
3. OR (lowest)

**Use Parentheses** for clarity: `WHERE (A OR B) AND C` is different from `WHERE A OR (B AND C)`

In [0]:
%sql
-- AND / OR / NOT — combine multiple conditions
-- Parentheses control evaluation order
-- AND: both conditions must be true
SELECT DriverID, FirstName, Age, Gender, Rating
FROM drivers
WHERE Gender = 'F'
  AND Rating >= 4.8
ORDER BY Rating DESC
LIMIT 5;


### Understanding AND:

In the query above:
- First condition: `Gender = 'F'` filters to only female drivers
- Second condition: `Rating >= 4.8` filters to high ratings
- `AND` means BOTH must be true: Female AND high-rated
- If we used `OR`, we'd get all females PLUS all high-rated drivers (male or female)

## Explanation: 

Special Operators

### BETWEEN (Inclusive Range)
```sql
WHERE column BETWEEN value1 AND value2
-- Equivalent to:
WHERE column >= value1 AND column <= value2
```
- **Inclusive**: Both endpoints are included
- Works with numbers, dates, strings
- Cleaner syntax than using >= AND <=

### IN (List Matching)
```sql
WHERE column IN (value1, value2, value3)
-- Equivalent to:
WHERE column = value1 OR column = value2 OR column = value3
```
- Matches ANY value in the list
- More readable than multiple ORs
- Can use with subqueries: `WHERE id IN (SELECT ...)`

### LIKE (Pattern Matching)
```sql
WHERE column LIKE 'pattern'
```

**Wildcards:**
- `%` : Matches any number of characters (including zero)
- `_` : Matches exactly one character

**Examples:**
- `'%toyota%'` : Contains "toyota" anywhere
- `'toyota%'` : Starts with "toyota"
- `'%toyota'` : Ends with "toyota"
- `'t___ta'` : 6 characters, starts with 't', ends with 'ta'
- `'%@gmail.com'` : Ends with "@gmail.com"

**Case Sensitivity**: LIKE is case-insensitive in many SQL dialects, but case-sensitive in some. Use `ILIKE` for guaranteed case-insensitive matching (PostgreSQL, Snowflake).

In [0]:
%sql

-- BETWEEN: inclusive range (equivalent to >= AND <=)
SELECT DriverID, FirstName, Age
FROM drivers
WHERE Age BETWEEN 30 AND 35
LIMIT 5;
-- IN: match against a list of values
SELECT TaxiID, Model, Status
FROM taxi
WHERE Model IN ('Toyota Camry', 'BMW 5 Series', 'Mercedes E-Class')
  AND Status = 'Active'
LIMIT 5;
-- LIKE: pattern matching (% = any chars, _ = single char)
SELECT DriverID, Email
FROM drivers
WHERE Email LIKE '%@yahoo%'
LIMIT 5;


## Explanation: 

NOT Operator

```sql
WHERE column NOT IN (values)
WHERE column NOT BETWEEN x AND y
WHERE column NOT LIKE 'pattern'
WHERE NOT EXISTS (subquery)
```

**Purpose**: Negates/inverts the condition
- `NOT IN`: Exclude values from list
- `NOT LIKE`: Exclude pattern matches
- `NOT BETWEEN`: Exclude range
- `NOT EXISTS`: Check for non-existence

**Examples:**
- `Status NOT IN ('Active')` : All statuses except 'Active'
- `Model NOT LIKE '%Toyota%'` : Exclude Toyota models
- `Age NOT BETWEEN 30 AND 40` : Ages < 30 or > 40
- `Email IS NOT NULL` : Only rows with email addresses

**Alternative Syntax:**
- `NOT IN` can be written as `!= ALL`
- `NOT EXISTS` is often more efficient than `NOT IN` with subqueries

In [0]:
%sql

-- NOT — negate conditions
-- Combines with IN, BETWEEN, LIKE, EXISTS

SELECT TaxiID, LicensePlate, Model, Status
FROM taxi
WHERE Status NOT IN ('Active')
  AND Model NOT LIKE '%Toyota%'
LIMIT 5;


---

# Aggregate Functions & GROUP BY

## Concept

**Aggregate functions** collapse multiple rows into a single summary value.

**GROUP BY** creates groups, and each aggregate runs independently per group.

**HAVING** filters the results AFTER aggregation (unlike WHERE which filters BEFORE).

### Query Execution Order:
```
FROM → WHERE → GROUP BY → HAVING → SELECT → ORDER BY → LIMIT
     ↑                    ↑
   Filter rows       Filter groups
```

### Key Differences:
- **WHERE**: Filters individual rows BEFORE grouping
- **HAVING**: Filters aggregated groups AFTER grouping
- **WHERE**: Cannot use aggregate functions
- **HAVING**: Can use aggregate functions

## Explanation: 

Aggregate Functions

```sql
SELECT 
    COUNT(*),              -- Count all rows
    COUNT(column),         -- Count non-NULL values
    COUNT(DISTINCT col),   -- Count unique values
    SUM(column),           -- Sum of values
    AVG(column),           -- Average
    MIN(column),           -- Minimum value
    MAX(column),           -- Maximum value
    STDDEV(column),        -- Standard deviation
    VARIANCE(column)       -- Variance
FROM table;
```

### Common Aggregate Functions:

**Counting:**
- `COUNT(*)`: Counts all rows (including NULLs and duplicates)
- `COUNT(column)`: Counts non-NULL values only
- `COUNT(DISTINCT column)`: Counts unique non-NULL values

**Mathematical:**
- `SUM(column)`: Total of all values (ignores NULLs)
- `AVG(column)`: Mean/average (ignores NULLs)
- `MIN(column)`: Smallest value
- `MAX(column)`: Largest value

**Statistical:**
- `STDDEV(column)` or `STDDEV_POP()`: Standard deviation
- `VARIANCE(column)` or `VAR_POP()`: Variance
- `STDDEV_SAMP()`, `VAR_SAMP()`: Sample versions

**String Aggregates:**
- `COLLECT_LIST(column)`: Array of values (with duplicates)
- `COLLECT_SET(column)`: Array of unique values

### NULL Handling:
- Most aggregates **ignore NULL values**
- `COUNT(*)` includes NULLs, `COUNT(column)` excludes them
- Empty result set: COUNT returns 0, others return NULL

In [0]:
%sql

-- Core aggregate functions: COUNT, SUM, AVG, MIN, MAX
-- Without GROUP BY, they summarize the entire table

SELECT
    COUNT(*) AS total_drivers,
    COUNT(DISTINCT Gender) AS gender_count,
    ROUND(AVG(Age), 1) AS avg_age,
    MIN(Age) AS youngest,
    MAX(Age) AS oldest,
    ROUND(AVG(Rating), 3) AS avg_rating,
    ROUND(STDDEV(Rating), 3) AS rating_stddev,
    SUM(YearsExperience) AS total_experience_years
FROM drivers;


###  Concepts:

**Without GROUP BY**: Aggregates summarize the ENTIRE table into **one row**

**NULL Handling Examples:**
- If 5 rows have NULL in Age: `COUNT(*)` = 5, `COUNT(Age)` = 0
- `AVG(Age)` calculates average of non-NULL values only
- `SUM(Age)` treats NULLs as 0 (actually, ignores them)

**ROUND() for Readability:**
- `ROUND(AVG(Age), 1)`: 1 decimal place (44.9)
- `ROUND(AVG(Rating), 3)`: 3 decimal places (4.497)
- Without ROUND: Raw values like 44.8625 or 4.4973825

## Explanation: 
GROUP BY

```sql
SELECT 
    grouping_column,
    AGGREGATE_FUNCTION(column)
FROM table
GROUP BY grouping_column
ORDER BY column;
```

### Key Rules:

**1. Golden Rule**: Every non-aggregated column in SELECT **must** be in GROUP BY
```sql
--  CORRECT:
SELECT Gender, COUNT(*) FROM drivers GROUP BY Gender;

-- WRONG (Gender not in GROUP BY):
SELECT Gender, COUNT(*) FROM drivers;

-- WRONG (Age not in GROUP BY):
SELECT Gender, Age, COUNT(*) FROM drivers GROUP BY Gender;
```

**2. How GROUP BY Works:**
- Splits data into groups based on unique values
- Each group gets one output row
- Aggregates run independently within each group

**3. Multiple Grouping Columns:**
- `GROUP BY col1, col2`: Creates groups for each unique **combination**
- Example: `GROUP BY Gender, AgeGroup` creates groups like (M, Young), (M, Old), (F, Young), (F, Old)

**4. Performance:**
- GROUP BY can be expensive (requires sorting or hashing)
- Filter with WHERE before grouping to reduce data volume
- Order columns by cardinality: low cardinality first

### Common Patterns:
```sql
-- Count by category
SELECT Gender, COUNT(*) FROM drivers GROUP BY Gender;

-- Average by group
SELECT Department, AVG(Salary) FROM employees GROUP BY Department;

-- Multiple aggregates
SELECT City, COUNT(*), AVG(Age), MAX(Rating) 
FROM drivers GROUP BY City;
```

In [0]:
%sql

-- GROUP BY — aggregate per group
-- Each unique Gender value becomes its own group

SELECT
    Gender,
    COUNT(*) AS count,
    ROUND(AVG(Age), 1) AS avg_age,
    ROUND(AVG(YearsExperience), 1) AS avg_exp,
    ROUND(AVG(Rating), 3) AS avg_rating,
    ROUND(MIN(Rating), 2) AS min_rating,
    ROUND(MAX(Rating), 2) AS max_rating
FROM drivers
GROUP BY Gender
ORDER BY count DESC;


### How GROUP BY Works:

**Step-by-step for the query above:**

1. **Split**: Divide 400 drivers into groups
   - Group 1: All rows where Gender = 'M' (319 drivers)
   - Group 2: All rows where Gender = 'F' (81 drivers)

2. **Aggregate**: Run functions independently in each group
   - For 'M' group: COUNT(*) = 319, AVG(Age) = 44.8, etc.
   - For 'F' group: COUNT(*) = 81, AVG(Age) = 44.5, etc.

3. **Return**: One row per group (2 rows total)

**Important**: Without GROUP BY, you'd get 1 row for entire table. With GROUP BY Gender, you get 2 rows (one per gender).

In [0]:
%sql

-- GROUP BY with multiple columns
-- Creates groups for each unique combination

SELECT
    Gender,
    CASE
        WHEN YearsExperience <= 5  THEN 'Junior'
        WHEN YearsExperience <= 15 THEN 'Mid-level'
        ELSE 'Senior+'
    END AS exp_level,
    COUNT(*) AS count,
    ROUND(AVG(Rating), 3) AS avg_rating
FROM drivers
GROUP BY Gender, exp_level
ORDER BY Gender, exp_level;


### Multi-Column GROUP BY:

**Groups Created:**
- (F, Junior)
- (F, Mid-level)
- (F, Senior+)
- (M, Junior)
- (M, Mid-level)
- (M, Senior+)

Each unique **combination** of Gender and exp_level gets its own group and its own output row.

**Note**: The CASE expression creates exp_level, which is then used in GROUP BY. This is allowed because the CASE is evaluated before GROUP BY in the execution order.

## Explanation: 

HAVING Clause

```sql
SELECT grouping_column, AGGREGATE()
FROM table
WHERE row_filter           -- Filters rows BEFORE grouping
GROUP BY grouping_column
HAVING aggregate_filter    -- Filters groups AFTER aggregation
ORDER BY column;
```

### WHERE vs HAVING:

| Aspect | WHERE | HAVING |
|--------|-------|--------|
| **When** | Before GROUP BY | After GROUP BY |
| **Filters** | Individual rows | Aggregated groups |
| **Can use aggregates?** |  No |  Yes |
| **Performance** | Faster (reduces data early) | Slower (after aggregation) |

### Examples:

```sql
-- CORRECT: Filter rows before grouping
WHERE Age > 30  -- Individual row condition

-- CORRECT: Filter groups after aggregation
HAVING COUNT(*) > 10  -- Group condition

-- WRONG: Cannot use aggregate in WHERE
WHERE COUNT(*) > 10  -- Error!

-- WRONG: Should use WHERE for row-level filters
HAVING Age > 30  -- Inefficient!
```

### Best Practice:
- Use **WHERE** for row-level filters (before grouping)
- Use **HAVING** for aggregate conditions (after grouping)
- This minimizes data processed during aggregation

In [0]:
%sql

-- HAVING — filter AFTER aggregation
-- WHERE filters rows; HAVING filters groups
-- Find join-years where we recruited more than 40 drivers
SELECT
    YEAR(JoinDate) AS join_year,
    COUNT(*) AS drivers_recruited,
    ROUND(AVG(Rating), 2) AS avg_rating
FROM drivers
GROUP BY join_year
HAVING COUNT(*) > 40
ORDER BY join_year;


#### How does the query execution flow?

1. **FROM drivers**: Start with drivers table (400 rows)
2. **GROUP BY join_year**: Create groups by year (9 groups for years 2015-2024)
3. **SELECT**: Calculate COUNT(*) and AVG(Rating) for each group
4. **HAVING COUNT(*) > 40**: Filter out groups with ≤40 drivers
5. **ORDER BY**: Sort remaining groups by year

**Result**: Only 4 years had more than 40 recruits, so only 4 rows in output.



### Comparison:

**WHERE vs HAVING**:
- WHERE filters individual rows before grouping
- HAVING filters aggregated groups after GROUP BY
- You cannot use aggregate functions in WHERE
- Always prefer WHERE for non-aggregate conditions (better performance)

# SQL Cheat Sheet

| Category | Command / Function | What It Does |
|----------|-------------------|-------------|
| **Select** | `SELECT col1, col2` | Choose specific columns |
| **Select** | `SELECT DISTINCT col` | Remove duplicate values |
| **Select** | `col AS alias` | Rename a column in output |
| **Select** | `CASE WHEN ... THEN ... ELSE ... END` | Conditional logic |
| **Filter** | `WHERE condition` | Filter rows before aggregation |
| **Filter** | `AND / OR / NOT` | Combine multiple conditions |
| **Filter** | `BETWEEN a AND b` | Inclusive range filter |
| **Filter** | `IN (val1, val2, ...)` | Match against a list |
| **Filter** | `LIKE 'pattern%'` | Pattern matching (% = any, _ = one) |
| **Aggregate** | `COUNT(*) / COUNT(col)` | Count rows / non-null values |
| **Aggregate** | `SUM(col) / AVG(col)` | Sum / average of values |
| **Aggregate** | `MIN(col) / MAX(col)` | Minimum / maximum value |
| **Aggregate** | `STDDEV(col) / VARIANCE(col)` | Standard deviation / variance |
| **Group** | `GROUP BY col1, col2` | Create groups for aggregation |
| **Group** | `HAVING condition` | Filter groups after aggregation |
| **Join** | `INNER JOIN ... ON` | Only matching rows from both tables |
| **Join** | `LEFT JOIN ... ON` | All left rows + matching right |
| **Join** | `CROSS JOIN` | Every combination (cartesian) |
| **Window** | `ROW_NUMBER() OVER (...)` | Sequential number per partition |
| **Window** | `RANK() / DENSE_RANK()` | Rank with/without gaps on ties |
| **Window** | `SUM() OVER (ORDER BY ...)` | Running total |
| **Window** | `LAG(col, n) / LEAD(col, n)` | Previous / next row value |
| **Window** | `NTILE(n) OVER (...)` | Divide into n equal buckets |
| **CTE** | `WITH name AS (SELECT ...)` | Named temporary result set |
| **Subquery** | `(SELECT ... ) in WHERE` | Scalar or list subquery |
| **Subquery** | `EXISTS (SELECT ...)` | Check for matching rows |
| **Perf** | `EXPLAIN [FORMATTED]` | Show Spark execution plan |
| **Perf** | Filter early + select cols | Reduce I/O and shuffle |

# Concuding Remarks

1. **Temporary views** give DataFrames a name that SQL can query — they store the query definition, not the data, and disappear when the session ends.

2. **SELECT → WHERE → GROUP BY → HAVING** pipeline is the backbone of every analytical query. Understand the execution order.

3. **JOINs** combine tables based on related columns:
   - `INNER JOIN`: Only matching rows
   - `LEFT JOIN`: All left rows + matches from right (NULLs where no match)
   - `CROSS JOIN`: Every combination (cartesian product)

4. **Window functions** (ROW_NUMBER, RANK, LAG, LEAD, NTILE) perform calculations across related rows WITHOUT collapsing groups like GROUP BY does.

5. **CTEs** (WITH ... AS) break complex queries into named, readable steps. You can chain multiple CTEs and they can reference each other.

6. **Subqueries** nest queries inside queries:
   - Scalar: Return single value
   - Correlated: Reference outer query columns
   - EXISTS: Check for existence of matching rows

7. **Performance Best Practices**:
   - Always select specific columns (column pruning)
   - Filter early with WHERE (predicate pushdown)
   - Use EXPLAIN to verify Spark is optimizing correctly
   - Avoid SELECT * in production

---

#  Resources

**Databricks Documentation:**
- [SQL Language Reference](https://docs.databricks.com/sql/language-manual/index.html)
- [Delta Lake Best Practices](https://docs.databricks.com/delta/best-practices.html)
- [Performance Tuning](https://docs.databricks.com/optimizations/index.html)

**Next Steps:**
- Try modifying these queries with your own data
- Experiment with different WHERE conditions
- Combine multiple concepts (JOINs + Window Functions + CTEs)
- Use EXPLAIN to understand query execution plans